In [0]:
try:
    landing_table = dbutils.widgets.get("landing_table")
    history_table = dbutils.widgets.get("history_table")
except Exception as e:
    print(f"Error initializing variables: {e}")
    raise

In [0]:
try:
    spark.sql(f"""
        MERGE INTO {history_table} tgt
        USING (
            SELECT *
            FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY _load_timestamp) AS rn
                FROM {landing_table}
            ) sub
            WHERE rn = 1
        ) src
        ON tgt.id = src.id
        AND tgt._modified_ts < src._load_timestamp
        AND tgt._active_flag = 1

        WHEN MATCHED THEN
            UPDATE SET
                tgt._active_flag = 0,
                tgt._modified_ts = src._load_timestamp
    """)
except Exception as e:
    print(f"Error updating {history_table}: {e}")
    raise

In [0]:
try:
    spark.sql(f"""
        MERGE INTO {history_table} tgt
        USING (
            SELECT *
            FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY _load_timestamp) AS rn
                FROM {landing_table}
            ) sub
            WHERE rn = 1
        ) src
        ON tgt.id = src.id
        AND tgt._created_ts = src._load_timestamp

        WHEN NOT MATCHED
        THEN
            INSERT (
                _id,
                pay_date,
                pay_amount,
                payer_name,
                payee_name,
                payee_npi,
                check_number,
                pay_method,
                create_date,
                total_claims,
                claim_type,
                claim_charges,
                claim_payments,
                provider_adjustments,
                create_mode,
                patient_resp,
                downloaded,
                trans_id,
                id,
                _file_name,
                _created_ts,
                _modified_ts,
                _active_flag
            )
            VALUES (
                src._id,
                src.pay_date,
                src.pay_amount,
                src.payer_name,
                src.payee_name,
                src.payee_npi,
                src.check_number,
                src.pay_method,
                src.create_date,
                src.total_claims,
                src.claim_type,
                src.claim_charges,
                src.claim_payments,
                src.provider_adjustments,
                src.create_mode,
                src.patient_resp,
                src.downloaded,
                src.trans_id,
                src.id,
                src._file_name,
                src._load_timestamp,
                src._load_timestamp,
                1
            );
    """)
except Exception as e:
    print(f"Error inserting into {history_table}: {e}")
    raise